# Anomaly Detection

In this exercise, you will implement the anomaly detection algorithm and apply it to detect failing servers on a network. 



# Outline
- [ 1 - Packages ](#1)
- [ 2 - Anomaly detection](#2)
  - [ 2.1 Problem Statement](#2.1)
  - [ 2.2  Dataset](#2.2)
  - [ 2.3 Gaussian distribution](#2.3)
    - [ Exercise 1](#ex01)
    - [ Exercise 2](#ex02)
  - [ 2.4 High dimensional dataset](#2.4)


# 异常检测

在本练习中，你将实现异常检测算法，并将其应用于检测网络中出现故障的服务器。

# 目录
- [ 1 - 依赖包 ](#1)
- [ 2 - 异常检测](#2)
  - [ 2.1 问题陈述](#2.1)
  - [ 2.2 数据集](#2.2)
  - [ 2.3 高斯分布](#2.3)
    - [ 练习 1](#ex01)
    - [ 练习 2](#ex02)
  - [ 2.4 高维数据集](#2.4)

<a name="1"></a>
## 1 - Packages 

First, let's run the cell below to import all the packages that you will need during this assignment.
- [numpy](www.numpy.org) is the fundamental package for working with matrices in Python.
- [matplotlib](http://matplotlib.org) is a famous library to plot graphs in Python.
- ``utils.py`` contains helper functions for this assignment. You do not need to modify code in this file.


## 1 - 工具包 

首先，运行下面的单元格以导入本次作业所需的所有工具包。
- [numpy](www.numpy.org) 是 Python 中用于处理矩阵的基础包。
- [matplotlib](http://matplotlib.org) 是 Python 中著名的绘图库。
- ``utils.py`` 包含本次作业的辅助函数。你不需要修改此文件中的代码。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *

%matplotlib inline

<a name="2"></a>
## 2 - Anomaly detection

<a name="2.1"></a>
### 2.1 Problem Statement

In this exercise, you will implement an anomaly detection algorithm to
detect anomalous behavior in server computers.

The dataset contains two features - 
   * throughput (mb/s) and 
   * latency (ms) of response of each server.

While your servers were operating, you collected $m=307$ examples of how they were behaving, and thus have an unlabeled dataset $\{x^{(1)}, \ldots, x^{(m)}\}$. 
* You suspect that the vast majority of these examples are “normal” (non-anomalous) examples of the servers operating normally, but there might also be some examples of servers acting anomalously within this dataset.

You will use a Gaussian model to detect anomalous examples in your
dataset. 
* You will first start on a 2D dataset that will allow you to visualize what the algorithm is doing.
* On that dataset you will fit a Gaussian distribution and then find values that have very low probability and hence can be considered anomalies. 
* After that, you will apply the anomaly detection algorithm to a larger dataset with many dimensions. 

<a name="2.2"></a>
### 2.2  Dataset

You will start by loading the dataset for this task. 
- The `load_data()` function shown below loads the data into the variables `X_train`, `X_val` and `y_val` 
    - You will use `X_train` to fit a Gaussian distribution 
    - You will use `X_val` and `y_val` as a cross validation set to select a threshold and determine anomalous vs normal examples

## 2 - 异常检测

### 2.1 问题陈述

在本练习中，你将实现一种异常检测算法，用于检测服务器计算机中的异常行为。

该数据集包含两个特征：
   * 每台服务器的吞吐量（mb/s）和
   * 响应延迟（ms）。

在服务器运行期间，你收集了 $m=307$ 个运行表现样本，从而得到了一个无标签数据集 $\{x^{(1)}, \ldots, x^{(m)}\}$。
* 你推测这些样本中的绝大多数都是服务器正常运行的“正常”（非异常）样本，但该数据集中也可能包含一些服务器行为异常的样本。

你将使用高斯模型来检测数据集中的异常样本。
* 你将首先从一个二维数据集开始，以便可视化算法的工作过程。
* 在该数据集上，你将拟合一个高斯分布，然后找出概率非常低的值，这些值因此可以被视为异常。
* 之后，你将把异常检测算法应用于一个维度更高的更大数据集。

### 2.2 数据集

你将从加载本任务的数据集开始。
- 下面展示的 `load_data()` 函数将数据加载到变量 `X_train`、`X_val` 和 `y_val` 中
    - 你将使用 `X_train` 来拟合高斯分布
    - 你将使用 `X_val` 和 `y_val` 作为交叉验证集，用于选择阈值并判断异常样本与正常样本

In [ ]:
# Load the dataset
X_train, X_val, y_val = load_data()

#### View the variables
Let's get more familiar with your dataset.  
- A good place to start is to just print out each variable and see what it contains.

The code below prints the first five elements of each of the variables

#### 查看变量
让我们进一步熟悉你的数据集。
- 一个很好的起点是直接打印出每个变量并查看其中包含的内容。

下面的代码打印出每个变量的前五个元素

In [ ]:
# Display the first five elements of X_train
print("The first 5 elements of X_train are:\n", X_train[:5])  

In [ ]:
# Display the first five elements of X_val
print("The first 5 elements of X_val are\n", X_val[:5])  

In [ ]:
# Display the first five elements of y_val
print("The first 5 elements of y_val are\n", y_val[:5])  

#### Check the dimensions of your variables

Another useful way to get familiar with your data is to view its dimensions.

The code below prints the shape of `X_train`, `X_val` and `y_val`.

#### 检查变量的维度

熟悉数据的另一种有用方法是查看其维度。

下面的代码打印出 `X_train`、`X_val` 和 `y_val` 的形状。

In [ ]:
print ('The shape of X_train is:', X_train.shape)
print ('The shape of X_val is:', X_val.shape)
print ('The shape of y_val is: ', y_val.shape)

#### Visualize your data

Before starting on any task, it is often useful to understand the data by visualizing it. 
- For this dataset, you can use a scatter plot to visualize the data (`X_train`), since it has only two properties to plot (throughput and latency)

- Your plot should look similar to the one below
<img src="images/figure1.png" width="500" height="500">

#### 可视化数据

在开始任何任务之前，通过可视化来理解数据通常很有帮助。
- 对于这个数据集，由于它只有两个属性可供绘制（吞吐量和延迟），你可以使用散点图来可视化数据（`X_train`）

- 你的图形应该与下图类似

In [ ]:
# Create a scatter plot of the data. To change the markers to blue "x",
# we used the 'marker' and 'c' parameters
plt.scatter(X_train[:, 0], X_train[:, 1], marker='x', c='b') 

# Set the title
plt.title("The first dataset")
# Set the y-axis label
plt.ylabel('Throughput (mb/s)')
# Set the x-axis label
plt.xlabel('Latency (ms)')
# Set axis range
plt.axis([0, 30, 0, 30])
plt.show()

<a name="2.3"></a>
### 2.3 Gaussian distribution

To perform anomaly detection, you will first need to fit a model to the data’s distribution.

* Given a training set $\{x^{(1)}, ..., x^{(m)}\}$ you want to estimate the Gaussian distribution for each
of the features $x_i$. 

* Recall that the Gaussian distribution is given by

   $$ p(x ; \mu,\sigma ^2) = \frac{1}{\sqrt{2 \pi \sigma ^2}}\exp^{ - \frac{(x - \mu)^2}{2 \sigma ^2} }$$

   where $\mu$ is the mean and $\sigma^2$ controls the variance.
   
* For each feature $i = 1\ldots n$, you need to find parameters $\mu_i$ and $\sigma_i^2$ that fit the data in the $i$-th dimension $\{x_i^{(1)}, ..., x_i^{(m)}\}$ (the $i$-th dimension of each example).

### 2.2.1 Estimating parameters for a Gaussian

**Implementation**: 

Your task is to complete the code in `estimate_gaussian` below.

### 2.3 高斯分布

要执行异常检测，你首先需要为数据的分布拟合一个模型。

* 给定训练集 $\{x^{(1)}, ..., x^{(m)}\}$，你希望为每个特征 $x_i$ 估计高斯分布。

* 回顾高斯分布的表达式为

   $$ p(x ; \mu,\sigma ^2) = \frac{1}{\sqrt{2 \pi \sigma ^2}}\exp^{ - \frac{(x - \mu)^2}{2 \sigma ^2} }$$

   其中 $\mu$ 是均值，$\sigma^2$ 控制方差。
   
* 对于每个特征 $i = 1\ldots n$，你需要找到参数 $\mu_i$ 和 $\sigma_i^2$，使其拟合第 $i$ 维的数据 $\{x_i^{(1)}, ..., x_i^{(m)}\}$（即每个样本的第 $i$ 维）。

### 2.2.1 估计高斯分布的参数

**实现**：

你的任务是补全下面 `estimate_gaussian` 中的代码。

<a name="ex01"></a>
### Exercise 1

Please complete the `estimate_gaussian` function below to calculate `mu` (mean for each feature in `X`)and `var` (variance for each feature in `X`). 

You can estimate the parameters, ($\mu_i$, $\sigma_i^2$), of the $i$-th
feature by using the following equations. To estimate the mean, you will
use:

$$\mu_i = \frac{1}{m} \sum_{j=1}^m x_i^{(j)}$$

and for the variance you will use:
$$\sigma_i^2 = \frac{1}{m} \sum_{j=1}^m (x_i^{(j)} - \mu_i)^2$$

If you get stuck, you can check out the hints presented after the cell below to help you with the implementation.

### 练习 1

请补全下面的 `estimate_gaussian` 函数，以计算 `mu`（`X` 中每个特征的均值）和 `var`（`X` 中每个特征的方差）。

你可以使用以下公式估计第 $i$ 个特征的参数（$\mu_i$, $\sigma_i^2$）。为了估计均值，你将使用：

$$\mu_i = \frac{1}{m} \sum_{j=1}^m x_i^{(j)}$$

而对于方差，你将使用：
$$\sigma_i^2 = \frac{1}{m} \sum_{j=1}^m (x_i^{(j)} - \mu_i)^2$$

如果你遇到困难，可以查看下方单元格后面提供的提示来帮助你完成实现。

In [ ]:
# UNQ_C1
# GRADED FUNCTION: estimate_gaussian

def estimate_gaussian(X): 
    """
    Calculates mean and variance of all features 
    in the dataset
    
    Args:
        X (ndarray): (m, n) Data matrix
    
    Returns:
        mu (ndarray): (n,) Mean of all features
        var (ndarray): (n,) Variance of all features
    """

    m, n = X.shape
    
    ### START CODE HERE ### 
    
    ### END CODE HERE ### 
        
    return mu, var

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
  
   * You can implement this function in two ways: 
      * 1 - by having two nested for loops - one looping over the **columns** of `X` (each feature) and then looping over each data point. 
      * 2 - in a vectorized manner by using `np.sum()` with `axis = 0` parameter (since we want the sum for each column)

    
   * Here's how you can structure the overall implementation of this function for the vectorized implementation:
     ```python  
    def estimate_gaussian(X): 
        m, n = X.shape
    
        ### START CODE HERE ### 
        mu = # Your code here to calculate the mean of every feature
        var = # Your code here to calculate the variance of every feature 
        ### END CODE HERE ### 
        
        return mu, var
    ```

    If you're still stuck, you can check the hints presented below to figure out how to calculate `mu` and `var`.
    
    <details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate mu</b></font></summary>
           &emsp; &emsp; You can use <a href="https://numpy.org/doc/stable/reference/generated/numpy.sum.html">np.sum</a> to with `axis = 0` parameter to get the sum for each column of an array
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; More hints to calculate mu</b></font></summary>
               &emsp; &emsp; You can compute mu as <code>mu = 1 / m * np.sum(X, axis = 0)</code>
           </details>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate var</b></font></summary>
           &emsp; &emsp; You can use <a href="https://numpy.org/doc/stable/reference/generated/numpy.sum.html">np.sum</a> to with `axis = 0` parameter to get the sum for each column of an array and <code>**2</code> to get the square.
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; More hints to calculate var</b></font></summary>
               &emsp; &emsp; You can compute var as <code> var = 1 / m * np.sum((X - mu) ** 2, axis = 0)</code>
           </details>
    </details>
    
</details>

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
  
   * 你可以通过两种方式实现此函数：
      * 1 - 使用两层嵌套的 for 循环：外层遍历 `X` 的**列**（每个特征），内层遍历每个数据点。
      * 2 - 以向量化的方式，结合 `axis = 0` 参数使用 `np.sum()`（因为我们要对每一列求和）。

    
   * 以下是该函数向量化实现的整体结构：
     ```python  
    def estimate_gaussian(X): 
        m, n = X.shape
    
        ### START CODE HERE ### 
        mu = # 在此处编写代码，计算每个特征的均值
        var = # 在此处编写代码，计算每个特征的方差 
        ### END CODE HERE ### 
        
        return mu, var
    ```

    如果你仍然卡住了，可以查看下面的提示，了解如何计算 `mu` 和 `var`。
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 mu 的提示</b></font></summary>
           &emsp; &emsp; 你可以配合 `axis = 0` 参数使用 <a href="https://numpy.org/doc/stable/reference/generated/numpy.sum.html">np.sum</a> 来获取数组每一列的和
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 关于计算 mu 的更多提示</b></font></summary>
               &emsp; &emsp; 你可以这样计算 mu：<code>mu = 1 / m * np.sum(X, axis = 0)</code>
           </details>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 var 的提示</b></font></summary>
           &emsp; &emsp; 你可以配合 `axis = 0` 参数使用 <a href="https://numpy.org/doc/stable/reference/generated/numpy.sum.html">np.sum</a> 来获取数组每一列的和，并使用 <code>**2</code> 来求平方。
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 关于计算 var 的更多提示</b></font></summary>
               &emsp; &emsp; 你可以这样计算 var：<code> var = 1 / m * np.sum((X - mu) ** 2, axis = 0)</code>
           </details>
    </details>
    
</details>

You can check if your implementation is correct by running the following test code:

你可以运行以下测试代码来检查你的实现是否正确：

In [ ]:
# Estimate mean and variance of each feature
mu, var = estimate_gaussian(X_train)              

print("Mean of each feature:", mu)
print("Variance of each feature:", var)
    
# UNIT TEST
from public_tests import *
estimate_gaussian_test(estimate_gaussian)

**Expected Output**:
<table>
  <tr>
    <td> <b>Mean of each feature: <b>  </td> 
    <td> [14.11222578 14.99771051]</td> 
   </tr>    
   <tr>
    <td> <b>Variance of each feature: <b>  </td>
     <td> [1.83263141 1.70974533] </td> 
  </tr>
</table>

**预期输出**：
<table>
  <tr>
    <td> <b>每个特征的均值：<b>  </td> 
    <td> [14.11222578 14.99771051]</td> 
   </tr>    
   <tr>
    <td> <b>每个特征的方差：<b>  </td>
     <td> [1.83263141 1.70974533] </td> 
  </tr>
</table>

Now that you have completed the code in `estimate_gaussian`, we will visualize the contours of the fitted Gaussian distribution. 

You should get a plot similar to the figure below. 
<img src="images/figure2.png" width="500" height="500">


From your plot you can see that most of the examples are in the region with the highest probability, while the anomalous examples are in the regions with lower probabilities.

既然你已经完成了 `estimate_gaussian` 中的代码，接下来我们将可视化拟合得到的高斯分布的等高线。

你应该会得到一个与下图类似的图形。

从图中你可以看到，大多数样本都位于概率最高的区域，而异常样本则位于概率较低的区域。

In [ ]:
# Returns the density of the multivariate normal
# at each data point (row) of X_train
p = multivariate_gaussian(X_train, mu, var)

#Plotting code 
visualize_fit(X_train, mu, var)

### 2.2.2 Selecting the threshold $\epsilon$

Now that you have estimated the Gaussian parameters, you can investigate which examples have a very high probability given this distribution and which examples have a very low probability.  

* The low probability examples are more likely to be the anomalies in our dataset. 
* One way to determine which examples are anomalies is to select a threshold based on a cross validation set. 

In this section, you will complete the code in `select_threshold` to select the threshold $\varepsilon$ using the $F_1$ score on a cross validation set.

* For this, we will use a cross validation set
$\{(x_{\rm cv}^{(1)}, y_{\rm cv}^{(1)}),\ldots, (x_{\rm cv}^{(m_{\rm cv})}, y_{\rm cv}^{(m_{\rm cv})})\}$, where the label $y=1$ corresponds to an anomalous example, and $y=0$ corresponds to a normal example. 
* For each cross validation example, we will compute $p(x_{\rm cv}^{(i)})$. The vector of all of these probabilities $p(x_{\rm cv}^{(1)}), \ldots, p(x_{\rm cv}^{(m_{\rm cv)}})$ is passed to `select_threshold` in the vector `p_val`. 
* The corresponding labels $y_{\rm cv}^{(1)}, \ldots, y_{\rm cv}^{(m_{\rm cv)}}$ is passed to the same function in the vector `y_val`.

### 2.2.2 选择阈值 $\epsilon$

既然你已经估计了高斯分布参数，接下来就可以研究在该分布下哪些样本具有很高的概率，哪些样本具有很低的概率。

* 概率低的样本更有可能是我们数据集中的异常值。
* 确定哪些样本是异常值的一种方法是基于交叉验证集选择一个阈值。

在本节中，你将补全 `select_threshold` 中的代码，利用交叉验证集上的 $F_1$ 分数来选择阈值 $\varepsilon$。

* 为此，我们将使用一个交叉验证集 $\{(x_{\rm cv}^{(1)}, y_{\rm cv}^{(1)}),\ldots, (x_{\rm cv}^{(m_{\rm cv})}, y_{\rm cv}^{(m_{\rm cv})})\}$，其中标签 $y=1$ 对应异常样本，$y=0$ 对应正常样本。
* 对于每个交叉验证样本，我们将计算 $p(x_{\rm cv}^{(i)})$。所有这些概率组成的向量 $p(x_{\rm cv}^{(1)}), \ldots, p(x_{\rm cv}^{(m_{\rm cv)}})$ 会通过向量 `p_val` 传递给 `select_threshold`。
* 对应的标签 $y_{\rm cv}^{(1)}, \ldots, y_{\rm cv}^{(m_{\rm cv)}}$ 会通过向量 `y_val` 传递给同一个函数。

<a name="ex02"></a>
### Exercise 2
Please complete the `select_threshold` function below to find the best threshold to use for selecting outliers based on the results from a validation set (`p_val`) and the ground truth (`y_val`). 

* In the provided code `select_threshold`, there is already a loop that will try many different values of $\varepsilon$ and select the best $\varepsilon$ based on the $F_1$ score. 

* You need implement code to calculate the F1 score from choosing `epsilon` as the threshold and place the value in `F1`. 

  * Recall that if an example $x$ has a low probability $p(x) < \varepsilon$, then it is classified as an anomaly. 
        
  * Then, you can compute precision and recall by: 
   $$\begin{aligned}
   prec&=&\frac{tp}{tp+fp}\\
   rec&=&\frac{tp}{tp+fn},
   \end{aligned}$$ where
    * $tp$ is the number of true positives: the ground truth label says it’s an anomaly and our algorithm correctly classified it as an anomaly.
    * $fp$ is the number of false positives: the ground truth label says it’s not an anomaly, but our algorithm incorrectly classified it as an anomaly.
    * $fn$ is the number of false negatives: the ground truth label says it’s an anomaly, but our algorithm incorrectly classified it as not being anomalous.

  * The $F_1$ score is computed using precision ($prec$) and recall ($rec$) as follows:
    $$F_1 = \frac{2\cdot prec \cdot rec}{prec + rec}$$ 

**Implementation Note:** 
In order to compute $tp$, $fp$ and $fn$, you may be able to use a vectorized implementation rather than loop over all the examples.


If you get stuck, you can check out the hints presented after the cell below to help you with the implementation.

### 练习 2
请补全下面的 `select_threshold` 函数，根据验证集的结果（`p_val`）和真实标签（`y_val`），找出用于筛选异常值的最佳阈值。

* 在提供的 `select_threshold` 代码中，已经有一个循环会尝试许多不同的 $\varepsilon$ 值，并根据 $F_1$ 分数选择最佳的 $\varepsilon$。

* 你需要编写代码，计算以 `epsilon` 作为阈值时的 F1 分数，并将结果存入 `F1`。

  * 回顾一下，如果一个样本 $x$ 的概率很低，满足 $p(x) < \varepsilon$，则它会被归类为异常。
        
  * 然后，你可以按以下公式计算精确率（precision）和召回率（recall）：
   $$\begin{aligned}
   prec&=&\frac{tp}{tp+fp}\\
   rec&=&\frac{tp}{tp+fn},
   \end{aligned}$$ 其中
    * $tp$ 是真正例（true positives）的数量：真实标签表明它是异常，且我们的算法正确地将其归类为异常。
    * $fp$ 是假正例（false positives）的数量：真实标签表明它不是异常，但我们的算法错误地将其归类为异常。
    * $fn$ 是假负例（false negatives）的数量：真实标签表明它是异常，但我们的算法错误地将其归类为非异常。

  * $F_1$ 分数使用精确率（$prec$）和召回率（$rec$）按如下方式计算：
    $$F_1 = \frac{2\cdot prec \cdot rec}{prec + rec}$$ 

**实现提示**：
为了计算 $tp$、$fp$ 和 $fn$，你可以使用向量化实现，而不必遍历所有样本。

如果你遇到困难，可以查看下方单元格后面提供的提示来帮助你完成实现。

In [ ]:
# UNQ_C2
# GRADED FUNCTION: select_threshold

def select_threshold(y_val, p_val): 
    """
    Finds the best threshold to use for selecting outliers 
    based on the results from a validation set (p_val) 
    and the ground truth (y_val)
    
    Args:
        y_val (ndarray): Ground truth on validation set
        p_val (ndarray): Results on validation set
        
    Returns:
        epsilon (float): Threshold chosen 
        F1 (float):      F1 score by choosing epsilon as threshold
    """ 

    best_epsilon = 0
    best_F1 = 0
    F1 = 0
    
    step_size = (max(p_val) - min(p_val)) / 1000
    
    for epsilon in np.arange(min(p_val), max(p_val), step_size):
    
        ### START CODE HERE ### 
        
        ### END CODE HERE ### 
        
        if F1 > best_F1:
            best_F1 = F1
            best_epsilon = epsilon
        
    return best_epsilon, best_F1

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>

   * Here's how you can structure the overall implementation of this function for the vectorized implementation:
     ```python  
    def select_threshold(y_val, p_val): 
        best_epsilon = 0
        best_F1 = 0
        F1 = 0
    
        step_size = (max(p_val) - min(p_val)) / 1000
    
        for epsilon in np.arange(min(p_val), max(p_val), step_size):
    
            ### START CODE HERE ### 
            predictions = # Your code here to calculate predictions for each example using epsilon as threshold
        
            tp = # Your code here to calculate number of true positives
            fp = # Your code here to calculate number of false positives
            fn = # Your code here to calculate number of false negatives
        
            prec = # Your code here to calculate precision
            rec = # Your code here to calculate recall
        
            F1 = # Your code here to calculate F1
            ### END CODE HERE ### 
        
            if F1 > best_F1:
                best_F1 = F1
                best_epsilon = epsilon
        
        return best_epsilon, best_F1
    ```

    If you're still stuck, you can check the hints presented below to figure out how to calculate each variable.
    
    <details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate predictions</b></font></summary>
           &emsp; &emsp; If an example  𝑥  has a low probability  $p(x) < \epsilon$ , then it is classified as an anomaly. To get predictions for each example (0/ False for normal and 1/True for anomaly), you can use <code>predictions = (p_val < epsilon)</code>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate tp, fp, fn</b></font></summary>
           &emsp; &emsp; 
        <ul>
          <li>If you have several binary values in an $n$-dimensional
binary vector, you can find out how many values in this vector are 0 by using:  `np.sum(v == 0)`</li>
          <li>You can also apply a logical *and* operator to such binary vectors. For instance,  `predictions` is a binary vector of the size of your number of cross validation set, where the $i$-th element is 1 if your algorithm considers $x_{\rm cv}^{(i)}$ an anomaly, and 0 otherwise. </li>
          <li>You can then, for example, compute the number of false positives using:  
<code>fp = sum((predictions == 1) & (y_val == 0))</code>.</li>
        </ul>
         <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; More hints to calculate tp, fn</b></font></summary>
               &emsp; &emsp;
             <ul>
              <li>You can compute tp as <code> tp = np.sum((predictions == 1) & (y_val == 1))</code></li>
              <li>You can compute tn as <code> fn = np.sum((predictions == 0) & (y_val == 1))</code></li>  
              </ul>
          </details>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate precision</b></font></summary>
           &emsp; &emsp; You can calculate precision as <code>prec = tp / (tp + fp)</code>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate recall</b></font></summary>
           &emsp; &emsp; You can calculate recall as <code>rec = tp / (tp + fn)</code>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate F1</b></font></summary>
           &emsp; &emsp; You can calculate F1 as <code>F1 = 2 * prec * rec / (prec + rec)</code>
    </details>
    
</details>

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>

   * 以下是该函数向量化实现的整体结构：
     ```python  
    def select_threshold(y_val, p_val): 
        best_epsilon = 0
        best_F1 = 0
        F1 = 0
    
        step_size = (max(p_val) - min(p_val)) / 1000
    
        for epsilon in np.arange(min(p_val), max(p_val), step_size):
    
            ### START CODE HERE ### 
            predictions = # 在此处编写代码，以 epsilon 为阈值计算每个样本的预测结果
        
            tp = # 在此处编写代码，计算真正例的数量
            fp = # 在此处编写代码，计算假正例的数量
            fn = # 在此处编写代码，计算假负例的数量
        
            prec = # 在此处编写代码，计算精确率
            rec = # 在此处编写代码，计算召回率
        
            F1 = # 在此处编写代码，计算 F1 分数
            ### END CODE HERE ### 
        
            if F1 > best_F1:
                best_F1 = F1
                best_epsilon = epsilon
        
        return best_epsilon, best_F1
    ```

    如果你仍然卡住了，可以查看下面的提示，了解如何计算每个变量。
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 predictions 的提示</b></font></summary>
           &emsp; &emsp; 如果一个样本 𝑥 的概率很低，满足 $p(x) < \epsilon$，则将其归类为异常。要获取每个样本的预测结果（0/False 表示正常，1/True 表示异常），可以使用 <code>predictions = (p_val < epsilon)</code>
    </details>
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 tp、fp、fn 的提示</b></font></summary>
           &emsp; &emsp; 
        <ul>
          <li>如果你有一个包含多个二值元素的 $n$ 维二值向量，可以使用 `np.sum(v == 0)` 来统计该向量中有多少个值为 0</li>
          <li>你还可以对这类二值向量应用逻辑*与*（and）运算符。例如，`predictions` 是一个大小与交叉验证集样本数相同的二值向量，如果算法认为 $x_{\rm cv}^{(i)}$ 是异常，则第 $i$ 个元素为 1，否则为 0。</li>
          <li>然后，例如你可以使用以下方式计算假正例的数量：
<code>fp = sum((predictions == 1) & (y_val == 0))</code>。</li>
        </ul>
         <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 关于计算 tp、fn 的更多提示</b></font></summary>
               &emsp; &emsp;
             <ul>
              <li>你可以这样计算 tp：<code> tp = np.sum((predictions == 1) & (y_val == 1))</code></li>
              <li>你可以这样计算 fn：<code> fn = np.sum((predictions == 0) & (y_val == 1))</code></li>  
              </ul>
          </details>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>计算 precision 的提示</b></font></summary>
           &emsp; &emsp; 你可以这样计算精确率：<code>prec = tp / (tp + fp)</code>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>计算 recall 的提示</b></font></summary>
           &emsp; &emsp; 你可以这样计算召回率：<code>rec = tp / (tp + fn)</code>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>计算 F1 的提示</b></font></summary>
           &emsp; &emsp; 你可以这样计算 F1：<code>F1 = 2 * prec * rec / (prec + rec)</code>
    </details>
    
</details>

You can check your implementation using the code below

你可以使用下面的代码检查你的实现

In [ ]:
p_val = multivariate_gaussian(X_val, mu, var)
epsilon, F1 = select_threshold(y_val, p_val)

print('Best epsilon found using cross-validation: %e' % epsilon)
print('Best F1 on Cross Validation Set: %f' % F1)
    
# UNIT TEST
select_threshold_test(select_threshold)


**Expected Output**:
<table>
  <tr>
    <td> <b>Best epsilon found using cross-validation: <b>  </td> 
    <td> 8.99e-05</td> 
   </tr>    
   <tr>
    <td> <b>Best F1 on Cross Validation Set: <b>  </td>
     <td> 0.875 </td> 
  </tr>
</table>

**预期输出**：
<table>
  <tr>
    <td> <b>使用交叉验证找到的最佳 epsilon：<b>  </td> 
    <td> 8.99e-05</td> 
   </tr>    
   <tr>
    <td> <b>交叉验证集上的最佳 F1：<b>  </td>
     <td> 0.875 </td> 
  </tr>
</table>

Now we will run your anomaly detection code and circle the anomalies in the plot (Figure 3 below).

<img src="images/figure3.png" width="500" height="500">

现在我们将运行你的异常检测代码，并在图中圈出异常点（如下方图 3 所示）。

In [ ]:
# Find the outliers in the training set 
outliers = p < epsilon

# Visualize the fit
visualize_fit(X_train, mu, var)

# Draw a red circle around those outliers
plt.plot(X_train[outliers, 0], X_train[outliers, 1], 'ro',
         markersize= 10,markerfacecolor='none', markeredgewidth=2)

<a name="2.4"></a>
### 2.4 High dimensional dataset

Now,  we will run the anomaly detection algorithm that you implemented on a more realistic and much harder dataset.

In this dataset, each example is described by 11 features, capturing many more properties of your compute servers.

Let's start by loading the dataset.

- The `load_data()` function shown below loads the data into variables `X_train_high`, `X_val_high` and `y_val_high`
    -  `_high` is meant to distinguish these variables from the ones used in the previous part
    - We will use `X_train_high` to fit Gaussian distribution 
    - We will use `X_val_high` and `y_val_high` as a cross validation set to select a threshold and determine anomalous vs normal examples

### 2.4 高维数据集

现在，我们将在一个更贴近实际且难度大得多的数据集上运行你实现的异常检测算法。

在这个数据集中，每个样本由 11 个特征描述，捕捉了计算服务器的更多属性。

让我们从加载数据集开始。

- 下面展示的 `load_data()` 函数将数据加载到变量 `X_train_high`、`X_val_high` 和 `y_val_high` 中
    - `_high` 用于将这些变量与前一部分中使用的变量区分开来
    - 我们将使用 `X_train_high` 来拟合高斯分布
    - 我们将使用 `X_val_high` 和 `y_val_high` 作为交叉验证集，用于选择阈值并判断异常样本与正常样本

In [ ]:
# load the dataset
X_train_high, X_val_high, y_val_high = load_data_multi()

#### Check the dimensions of your variables

Let's check the dimensions of these new variables to become familiar with the data

#### 检查变量的维度

让我们检查这些新变量的维度，以便熟悉数据

In [ ]:
print ('The shape of X_train_high is:', X_train_high.shape)
print ('The shape of X_val_high is:', X_val_high.shape)
print ('The shape of y_val_high is: ', y_val_high.shape)

#### Anomaly detection 

Now, let's run the anomaly detection algorithm on this new dataset.

The code below will use your code to 
* Estimate the Gaussian parameters ($\mu_i$ and $\sigma_i^2$)
* Evaluate the probabilities for both the training data `X_train_high` from which you estimated the Gaussian parameters, as well as for the the cross-validation set `X_val_high`. 
* Finally, it will use `select_threshold` to find the best threshold $\varepsilon$. 

#### 异常检测

现在，让我们在这个新数据集上运行异常检测算法。

下面的代码将使用你编写的代码来：
* 估计高斯分布参数（$\mu_i$ 和 $\sigma_i^2$）
* 评估训练数据 `X_train_high`（你从中估计了高斯参数）以及交叉验证集 `X_val_high` 的概率。
* 最后，它将使用 `select_threshold` 找到最佳阈值 $\varepsilon$。

In [ ]:
# Apply the same steps to the larger dataset

# Estimate the Gaussian parameters
mu_high, var_high = estimate_gaussian(X_train_high)

# Evaluate the probabilites for the training set
p_high = multivariate_gaussian(X_train_high, mu_high, var_high)

# Evaluate the probabilites for the cross validation set
p_val_high = multivariate_gaussian(X_val_high, mu_high, var_high)

# Find the best threshold
epsilon_high, F1_high = select_threshold(y_val_high, p_val_high)

print('Best epsilon found using cross-validation: %e'% epsilon_high)
print('Best F1 on Cross Validation Set:  %f'% F1_high)
print('# Anomalies found: %d'% sum(p_high < epsilon_high))

**Expected Output**:
<table>
  <tr>
    <td> <b>Best epsilon found using cross-validation: <b>  </td> 
    <td> 1.38e-18</td> 
   </tr>    
   <tr>
    <td> <b>Best F1 on Cross Validation Set: <b>  </td>
     <td> 0.615385 </td> 
  </tr>
    <tr>
    <td> <b># anomalies found: <b>  </td>
     <td>  117 </td> 
  </tr>
</table>

**预期输出**：
<table>
  <tr>
    <td> <b>使用交叉验证找到的最佳 epsilon：<b>  </td> 
    <td> 1.38e-18</td> 
   </tr>    
   <tr>
    <td> <b>交叉验证集上的最佳 F1：<b>  </td>
     <td> 0.615385 </td> 
  </tr>
    <tr>
    <td> <b>发现的异常数量：<b>  </td>
     <td>  117 </td> 
  </tr>
</table>